In [1]:
import torch
from einops import einsum, rearrange
from jaxtyping import Float, Int
from torch import Tensor

from src.modules.attention import softmax

In [2]:
batch_size, seq_len, vocab_size = 4, 12, 100
torch.manual_seed(42)


logits = torch.randn(size=(batch_size, seq_len, vocab_size))
targets = torch.randint(low=0, high=vocab_size, size=(batch_size, seq_len))

In [3]:
predicted_soft = softmax(logits, dim=-1)

predicted_soft.shape, targets.shape

(torch.Size([4, 12, 100]), torch.Size([4, 12]))

In [4]:
predicted = torch.gather(logits, dim=-1, index=targets.unsqueeze(-1)).squeeze(
    -1
)

predicted.shape

torch.Size([4, 12])

In [5]:
predicted_2 = torch.take_along_dim(
    logits, targets.unsqueeze(-1), dim=-1
).squeeze(-1)

predicted_2.shape

torch.Size([4, 12])

In [6]:
targets.unsqueeze(-1).expand(logits.shape)

tensor([[[54, 54, 54,  ..., 54, 54, 54],
         [82, 82, 82,  ..., 82, 82, 82],
         [22, 22, 22,  ..., 22, 22, 22],
         ...,
         [15, 15, 15,  ..., 15, 15, 15],
         [52, 52, 52,  ..., 52, 52, 52],
         [83, 83, 83,  ..., 83, 83, 83]],

        [[13, 13, 13,  ..., 13, 13, 13],
         [19, 19, 19,  ..., 19, 19, 19],
         [95, 95, 95,  ..., 95, 95, 95],
         ...,
         [92, 92, 92,  ..., 92, 92, 92],
         [80, 80, 80,  ..., 80, 80, 80],
         [89, 89, 89,  ..., 89, 89, 89]],

        [[76, 76, 76,  ..., 76, 76, 76],
         [33, 33, 33,  ..., 33, 33, 33],
         [ 3,  3,  3,  ...,  3,  3,  3],
         ...,
         [37, 37, 37,  ..., 37, 37, 37],
         [18, 18, 18,  ..., 18, 18, 18],
         [65, 65, 65,  ..., 65, 65, 65]],

        [[ 1,  1,  1,  ...,  1,  1,  1],
         [58, 58, 58,  ..., 58, 58, 58],
         [63, 63, 63,  ..., 63, 63, 63],
         ...,
         [52, 52, 52,  ..., 52, 52, 52],
         [26, 26, 26,  ..., 26, 26, 

In [14]:
max_logits = logits.max(-1, keepdim=True)[0]
norm_logits = logits - max_logits
predicted = torch.gather(norm_logits, -1, targets.unsqueeze(-1)).squeeze(-1)

predicted.shape

torch.Size([4, 12])

In [15]:
norm_logits.shape, targets.unsqueeze(-1).shape

(torch.Size([4, 12, 100]), torch.Size([4, 12, 1]))

In [16]:
sum_logits = torch.exp(norm_logits).sum(-1)

ce_loss = predicted - torch.log(sum_logits)

predicted.shape

torch.Size([4, 12])

In [20]:
ce_loss.shape, ce_loss.mean(0).shape

(torch.Size([4, 12]), torch.Size([12]))

In [21]:
torch.exp(ce_loss.mean(0).mean())

tensor(0.0065)

In [17]:
torch.allclose(predicted, predicted_2)

False

In [31]:
def cross_entropy_loss(
    logits: Float[Tensor, "... seq_len vocab_size"],
    targets: Float[Tensor, "... seq_len"],
) -> Float[Tensor, "... seq_len"]:
    """Compute the cross entropy loss with respect to
    the logits outputed by the network.

    Returns:
        Float[Tensor, '... seq_len']: CEL averaged along the batch.
    """
    max_logits = logits.max(-1, keepdim=True)[0]
    norm_logits = logits - max_logits
    predicted = torch.gather(norm_logits, -1, targets.unsqueeze(-1)).squeeze(
        -1
    )

    sum_logits = torch.exp(norm_logits).sum(-1)

    ce_loss = -(predicted - torch.log(sum_logits))

    return ce_loss.mean(0)


In [32]:
inputs = torch.tensor(
    [
        [
            [0.1088, 0.1060, 0.6683, 0.5131, 0.0645],
            [0.4538, 0.6852, 0.2520, 0.3792, 0.2675],
            [0.4578, 0.3357, 0.6384, 0.0481, 0.5612],
            [0.9639, 0.8864, 0.1585, 0.3038, 0.0350],
        ],
        [
            [0.3356, 0.9013, 0.7052, 0.8294, 0.8334],
            [0.6333, 0.4434, 0.1428, 0.5739, 0.3810],
            [0.9476, 0.5917, 0.7037, 0.2987, 0.6208],
            [0.8541, 0.1803, 0.2054, 0.4775, 0.8199],
        ],
    ]
)
targets = torch.tensor([[1, 0, 2, 2], [4, 1, 4, 0]])

inputs.size(-1)

5

In [37]:
(inputs.shape, inputs.view(-1, inputs.size(-1)).shape, targets.view(-1).shape)

(torch.Size([2, 4, 5]), torch.Size([8, 5]), torch.Size([8]))

In [ ]:
cross_entropy_loss(inputs.view(-1, inputs.size(-1)), targets.view(-1))

tensor(1.6095)

In [38]:
cross_entropy_loss(inputs, targets)

tensor([1.6718, 1.5956, 1.5211, 1.6494])

In [80]:
import math
from collections.abc import Callable, Iterable
from typing import Optional

import torch


class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()

        reductions = []
        for group in self.param_groups:
            lr = group["lr"]  # Get the learning rate.
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]  # Get state associated with p.
                t = state.get(
                    "t", 0
                )  # Get iteration number from the state, or initial value.
                grad = (
                    p.grad.data
                )  # Get the gradient of loss with respect to p.
                reduction = (
                    lr / math.sqrt(t + 1) * grad
                )  # Update weight tensor in-place.
                reductions.append(reduction)
                p.data -= reduction
                state["t"] = t + 1  # Increment iteration number.
        print(f"Mean reductions: {torch.mean(reductions[0])}")
        return loss

In [81]:
def toy_training(lr=1):
    weights = torch.nn.Parameter(5 * torch.randn((10, 10))).to(torch.float32)
    opt = SGD([weights], lr=lr)
    for t in range(10):
        opt.zero_grad()  # Reset the gradients for all learnable parameters.
        loss = (weights**2).mean()  # Compute a scalar loss value.
        loss.backward()  # Run backward pass, which computes gradients.
        opt.step()  # Run optimizer step.

        print(f"Iteration {t} loss: {loss.cpu().item()}")

In [82]:
toy_training(lr=1)

Mean reductions: 0.019561519846320152
Iteration 0 loss: 27.618986129760742
Mean reductions: 0.013555441051721573
Iteration 1 loss: 26.5252742767334
Mean reductions: 0.010911447927355766
Iteration 2 loss: 25.780332565307617
Mean reductions: 0.00934047531336546
Iteration 3 loss: 25.188398361206055
Mean reductions: 0.00827083084732294
Iteration 4 loss: 24.687150955200195
Mean reductions: 0.007482671178877354
Iteration 5 loss: 24.247509002685547
Mean reductions: 0.006871042773127556
Iteration 6 loss: 23.853160858154297
Mean reductions: 0.006378686986863613
Iteration 7 loss: 23.4939022064209
Mean reductions: 0.005971359089016914
Iteration 8 loss: 23.162818908691406
Mean reductions: 0.0056271618232131
Iteration 9 loss: 22.855012893676758


In [83]:
toy_training(lr=10)

Mean reductions: -0.007680471055209637
Iteration 0 loss: 28.700780868530273
Mean reductions: -0.004344722256064415
Iteration 1 loss: 18.368501663208008
Mean reductions: -0.003045768244192004
Iteration 2 loss: 13.540474891662598
Mean reductions: -0.002333134412765503
Iteration 3 loss: 10.593975067138672
Mean reductions: -0.0018781375838443637
Iteration 4 loss: 8.581120491027832
Mean reductions: -0.0015611463459208608
Iteration 5 loss: 7.114731311798096
Mean reductions: -0.0013273278018459678
Iteration 6 loss: 6.000331878662109
Mean reductions: -0.0011477470397949219
Iteration 7 loss: 5.12745475769043
Mean reductions: -0.0010055905440822244
Iteration 8 loss: 4.427960395812988
Mean reductions: -0.000890387746039778
Iteration 9 loss: 3.857245683670044


In [84]:
toy_training(lr=100)

Mean reductions: -1.3320525884628296
Iteration 0 loss: 21.264179229736328
Mean reductions: 0.9419034719467163
Iteration 1 loss: 21.264175415039062
Mean reductions: -0.31855544447898865
Iteration 2 loss: 3.648355007171631
Mean reductions: 0.04267832264304161
Iteration 3 loss: 0.08731335401535034
Mean reductions: 9.756514485204093e-10
Iteration 4 loss: 1.0718002866487214e-16
Mean reductions: 9.40277924965649e-11
Iteration 5 loss: 1.1945878742670574e-18
Mean reductions: 1.5974492562076392e-11
Iteration 6 loss: 4.022598059656127e-20
Mean reductions: 3.647097814724054e-12
Iteration 7 loss: 2.3962893240874862e-21
Mean reductions: 1.0071183253676508e-12
Iteration 8 loss: 2.0556917493912208e-22
Mean reductions: 3.184787774213743e-13
Iteration 9 loss: 2.284101943768023e-23


In [85]:
toy_training(lr=1000)

Mean reductions: -1.6182866096496582
Iteration 0 loss: 28.801185607910156
Mean reductions: 21.741743087768555
Iteration 1 loss: 10397.2275390625
Mean reductions: -233.2999267578125
Iteration 2 loss: 1795764.75
Mean reductions: 2130.955322265625
Iteration 3 loss: 199759664.0
Mean reductions: -17153.86328125
Iteration 4 loss: 16180531200.0
Mean reductions: 124401.2890625
Iteration 5 loss: 1021176905728.0
Mean reductions: -825211.75
Iteration 6 loss: 52423884275712.0
Mean reductions: 5063216.0
Iteration 7 loss: 2255499659575296.0
Mean reductions: -28981136.0
Iteration 8 loss: 8.313290594503885e+16
Mean reductions: 155798752.0
Iteration 9 loss: 2.6694899874261893e+18


In [ ]:
grad = torch.ones((2, 2)) * 4

grad

tensor([[4., 4.],
        [4., 4.]])

In [ ]:
grad**2

tensor([[16., 16.],
        [16., 16.]])

In [ ]:
1e2

2000.0